# TASK-4

# ── CELL 1: Configuration ──

In [0]:
import os
def get_path(key, default=None):
    value = os.getenv(key, default)
    if value is None:
        raise ValueError(f"Missing environment variable: {key}")
    return value
VOLUME_PATH = get_path("INPUT_PATH", default="/default/input/path")
OUTPUT_PATH = get_path("OUTPUT_PATH", default="/default/output/path")
OUTPUT_P_PATH = get_path("OUTPUT_P_PATH", default="/default/output/parquet/path")

In [0]:
from dotenv import load_dotenv
import os
 
load_dotenv(".env", override=True)
OUTPUT_P_PATH = os.getenv("OUTPUT_P_PATH")
INPUT_PATH = os.getenv("INPUT_PATH")
OUTPUT_PATH = os.getenv("OUTPUT_PATH")
 

In [0]:
files = dbutils.fs.ls(VOLUME_PATH)
for f in files:
    print(f.name)

# ── CELL 2: Imports ──

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

print(" Imports done")

# ── CELL 3: Ingestion ───
# Columns to drop — junk image/promo cols in only 1-2 files

In [0]:

JUNK_COLS = {
    "blackfridaybelts_bg_src",
    "blackfridaybelts_content",
    "product_locatelabels_img_src"
}

def standardize_col_name(col):
    """Lowercase, strip, replace spaces and hyphens with underscores."""
    return col.strip().lower().replace(" ", "_").replace("-", "_")

def load_file(path):
    """
    Load one CSV with correct options to handle:
    - Quoted fields containing commas (multiLine + escape)
    - All columns as String (we cast explicitly later)
    - Extract category from filename
    """
    filename = path.split("/")[-1]
    category = filename.replace("us-shein-", "")
    category = "-".join(category.split("-")[:-1])
    
    df = spark.read \
        .option("header", True) \
        .option("inferSchema", False) \
        .option("multiLine", True) \
        .option("quote", '"') \
        .option("escape", '"') \
        .option("mode", "PERMISSIVE") \
        .csv(path)

    # Standardize column names immediately after load
    df = df.toDF(*[standardize_col_name(c) for c in df.columns])

    # Drop junk columns if present
    cols_to_drop = [c for c in df.columns if c in JUNK_COLS]
    if cols_to_drop:
        df = df.drop(*cols_to_drop)

    # Add category
    df = df.withColumn("category_name", F.lit(category))
    return df, category

def load_all_files(volume_path):
    files = dbutils.fs.ls(volume_path)
    csv_files = [f.path for f in files if f.name.endswith(".csv")]
    dataframes = []
    for path in csv_files:
        df, category = load_file(path)
        dataframes.append(df)
    return dataframes

raw_dfs = load_all_files(VOLUME_PATH)

In [0]:
print("="*80)
print("1. INGESTION STAGE - RAW FILES")
print("="*80)
print(f"\nTotal files loaded: {len(raw_dfs)}\n")

for idx, df in enumerate(raw_dfs, 1):
    category = df.select("category_name").first()[0]
    row_count = df.count()
    col_count = len(df.columns)
    print(f"{idx:2d}. {category:<45} | Rows: {row_count:>6,} | Cols: {col_count:>2}")

print(f"\n{'='*80}")
total_raw_rows = sum(df.count() for df in raw_dfs)
print(f"Total rows ingested: {total_raw_rows:,}")
print(f"Total files: {len(raw_dfs)}")
print(f"{'='*80}\n")

In [0]:
print("="*80)
print("JUNK COLUMNS DROPPED DURING INGESTION")
print("="*80)
print("\nColumns defined as junk (dropped if present in source files):")
JUNK_COLS_LIST = [
    "blackfridaybelts_bg_src",
    "blackfridaybelts_content",
    "product_locatelabels_img_src"
]
for idx, col in enumerate(JUNK_COLS_LIST, 1):
    print(f"  {idx}. {col}")
print(f"\nTotal junk columns: {len(JUNK_COLS_LIST)}")
print(f"{'='*80}\n")

# ── CELL 4: Cleaning ──

In [0]:

def clean_empty_strings(df):
    """Replace empty strings with null across all columns."""
    for col_name in df.columns:
        df = df.withColumn(col_name,
            F.when(F.col(col_name) == "", None)
             .otherwise(F.col(col_name))
        )
    return df

def drop_bad_price_rows(df):
    """
    Drop rows where price is not null but doesn't start with $.
    These are CSV-shifted rows where a title fragment landed in price.
    """
    if "price" in df.columns:
        df = df.filter(
            F.col("price").isNull() |
            F.col("price").rlike(r"^\$[\d,\.]+$")
        )
    return df

def clean_df(df):
    df = clean_empty_strings(df)
    df = drop_bad_price_rows(df)
    return df

cleaned_dfs = [clean_df(df) for df in raw_dfs]

In [0]:
print("="*80)
print("2. CLEANING STAGE")
print("="*80)

total_cleaned_rows = sum(df.count() for df in cleaned_dfs)
rows_removed_cleaning = sum(raw_dfs[i].count() - cleaned_dfs[i].count() for i in range(len(raw_dfs)))

print(f"\nRows after cleaning: {total_cleaned_rows:,}")
print(f"Rows removed during cleaning: {rows_removed_cleaning:,}")
print(f"\nColumn count remains same at each file (transformations come next)")
print(f"{'='*80}\n")

# ── CELL 5: Transformation ──

In [0]:

def transform_price(df):
    """$2.03 → 2.03 (Double) → price_usd"""
    if "price" in df.columns:
        df = df.withColumn("price_usd",
            F.expr("try_cast(regexp_replace(price, '[\\$,]', '') as DOUBLE)")
        ).drop("price")
    else:
        df = df.withColumn("price_usd", F.lit(None).cast(DoubleType()))
    return df

def transform_discount(df):
    """-22% → 22.0 (Double) → pct_discount. Null means no discount → 0.0"""
    if "discount" in df.columns:
        df = df.withColumn("pct_discount",
            F.expr("try_cast(regexp_replace(discount, '[\\-%]', '') as DOUBLE)")
        ).drop("discount")
    else:
        df = df.withColumn("pct_discount", F.lit(None).cast(DoubleType()))
    df = df.withColumn("pct_discount",
        F.when(F.col("pct_discount").isNull(), 0.0)
         .otherwise(F.col("pct_discount")))
    return df

def transform_qty_sold(df):
    """'10k+ sold recently' → 10000, '500+ sold recently' → 500 → qty_sold"""
    if "selling_proposition" in df.columns:
        df = df.withColumn("qty_sold",
            F.when(
                F.col("selling_proposition").rlike(r"(?i)[0-9]+\.?[0-9]*k\+?\s*sold"),
                (F.expr("try_cast(regexp_extract(selling_proposition,'([0-9]+\\.?[0-9]*)',1) as DOUBLE)") * 1000)
            ).when(
                F.col("selling_proposition").rlike(r"[0-9]+\+?\s*sold"),
                F.expr("try_cast(regexp_extract(selling_proposition,'([0-9]+)',1) as DOUBLE)")
            ).otherwise(None)
        )
        df = df.withColumn("qty_sold",
            F.col("qty_sold").cast(IntegerType())
        ).drop("selling_proposition")
    else:
        df = df.withColumn("qty_sold", F.lit(None).cast(IntegerType()))
    df = df.withColumn("qty_sold",
        F.when(F.col("qty_sold").isNull(), 0).otherwise(F.col("qty_sold")))
    return df

def transform_rank(df):
    """'#1 Best Seller' → 1 → rank. Unranked → 0"""
    if "rank_title" in df.columns:
        df = df.withColumn("rank",
            F.when(
                F.col("rank_title").rlike(r"#\d+"),
                F.regexp_extract(F.col("rank_title"), r"#(\d+)", 1).cast(IntegerType())
            ).otherwise(None)
        ).drop("rank_title")
    else:
        df = df.withColumn("rank", F.lit(None).cast(IntegerType()))
    df = df.withColumn("rank",
        F.when(F.col("rank").isNull(), 0).otherwise(F.col("rank")))
    return df

def transform_color_count(df):
    """Cast color_count from String to Integer. Null → 0"""
    if "color_count" in df.columns:
        df = df.withColumn("color_count",
            F.expr("try_cast(color_count as INT)")
        )
    else:
        df = df.withColumn("color_count", F.lit(None).cast(IntegerType()))
    df = df.withColumn("color_count",
        F.when(F.col("color_count").isNull(), 0).otherwise(F.col("color_count")))
    return df

def transform_df(df):
    df = transform_price(df)
    df = transform_discount(df)
    df = transform_qty_sold(df)
    df = transform_rank(df)
    df = transform_color_count(df)
    return df

transformed_dfs = [transform_df(df) for df in cleaned_dfs]

In [0]:
print("="*80)
print("3. TRANSFORMATION STAGE")
print("="*80)

# Show column changes during transformation
print("\nColumn transformations applied:")
print("  - 'price' → 'price_usd' (string to double)")
print("  - 'discount' → 'pct_discount' (string to double)")
print("  - 'selling_proposition' → 'qty_sold' (string to integer)")
print("  - 'rank_title' → 'rank' (string to integer)")
print("  - 'color_count' → type cast to integer")

print("\nSample transformed schema from first dataframe:")
transformed_dfs[0].printSchema()

total_transformed_rows = sum(df.count() for df in transformed_dfs)
print(f"\nTotal rows after transformation: {total_transformed_rows:,}")
print(f"Column count per file: {len(transformed_dfs[0].columns)} (varies by file based on source columns)")
print(f"{'='*80}\n")

# ── CELL 6: Merge ──

In [0]:

def merge_all(dfs):
    # Collect all unique columns across all files
    all_cols = sorted(set(c for df in dfs for c in df.columns))

    aligned = []
    for df in dfs:
        for col in all_cols:
            if col not in df.columns:
                # Add missing col as null with correct type
                df = df.withColumn(col, F.lit(None).cast("string"))
        # Always select in consistent order by name — never by position
        aligned.append(df.select(all_cols))

    merged = aligned[0]
    for df in aligned[1:]:
        merged = merged.unionByName(df)

    return merged

merged_df = merge_all(transformed_dfs)
merged_df.printSchema()

In [0]:
print("="*80)
print("4. MERGING STAGE")
print("="*80)

merged_row_count = merged_df.count()
merged_col_count = len(merged_df.columns)

print(f"\nRows after merging: {merged_row_count:,}")
print(f"Columns after merging: {merged_col_count}")
print(f"\nMerged columns: {', '.join(merged_df.columns)}")
print(f"\n{'='*80}\n")

# ── CELL 7: Consolidate + QC + Save ──

In [0]:

# ── Step 1: Consolidate product title ─────────────────────────────────────────
final_df = merged_df.withColumn(
    "product_title",
    F.coalesce(F.col("goods_title_link__jump"), F.col("goods_title_link"))
).drop("goods_title_link__jump", "goods_title_link")

# Rename URL column
final_df = final_df.withColumnRenamed(
    "goods_title_link__jump_href", "product_url"
)

# ── Step 2: Drop duplicates ───────────────────────────────────────────────────
final_df = final_df.dropDuplicates()

# ── Step 3: Drop null price rows (bad parse) ────────────────────────────────────
final_df = final_df.filter(F.col("price_usd").isNotNull())

# ── Step 4: Drop null product title rows ───────────────────────────────────
final_df = final_df.filter(F.col("product_title").isNotNull())

# ── Step 5: Reorder columns logically ──────────────────────────────────────
final_df = final_df.select(
    "category_name",
    "product_title",
    "price_usd",
    "pct_discount",
    "qty_sold",
    "rank",
    "rank_sub",
    "color_count",
)

# ── Step 6: Quality checks ─────────────────────────────────────────────
print("\n" + "="*60)
print("DATA QUALITY REPORT")
print("="*60)
print(f"Final rows: {final_df.count():,}")
print(f"Final columns: {len(final_df.columns)}")
print("\nNull check:")
for col in final_df.columns:
    n = final_df.filter(F.col(col).isNull()).count()
    print(f"  {col:<30} → {n:,} nulls")

print("\nValue ranges:")
ps = final_df.select(F.min("price_usd"), F.max("price_usd"), F.round(F.avg("price_usd"),2)).first()
ds = final_df.select(F.min("pct_discount"), F.max("pct_discount")).first()
qs = final_df.select(F.min("qty_sold"), F.max("qty_sold")).first()
rs = final_df.select(F.min("rank"), F.max("rank")).first()
cs = final_df.select(F.min("color_count"), F.max("color_count")).first()

print(f"  price_usd    → min: ${ps[0]}  max: ${ps[1]}  avg: ${ps[2]}")
print(f"  pct_discount → min: {ds[0]}%  max: {ds[1]}%")
print(f"  qty_sold     → min: {qs[0]}   max: {qs[1]:,}")
print(f"  rank         → min: {rs[0]}   max: {rs[1]}")
print(f"  color_count  → min: {cs[0]}   max: {cs[1]}")

print("\nRecords per category:")
category_counts = final_df.groupBy("category_name").count().orderBy(F.desc("count")).collect()
for row in category_counts:
    print(f"  {row['category_name']:<40} : {row['count']:>6,} rows")

print("\n" + "="*60)
final_df.printSchema()
print("\nSample rows:")
final_df.show(5, truncate=60)

# ── Step 7: Save ───────────────────────────────────────────────────────
final_df.coalesce(1) \
    .write.mode("overwrite") \
    .option("header", True) \
    .csv(OUTPUT_PATH)

print(f"\n✓ CSV saved to {OUTPUT_PATH}")

In [0]:
print("\n" + "="*80)
print("COMPLETE PIPELINE ANALYSIS SUMMARY")
print("="*80)

# Calculate all stages
total_raw_rows = sum(df.count() for df in raw_dfs)
total_cleaned_rows = sum(df.count() for df in cleaned_dfs)
total_transformed_rows = sum(df.count() for df in transformed_dfs)
merged_row_count = merged_df.count()
final_row_count = final_df.count()

print(f"\n{'Stage':<30} | {'Rows':<15} | {'Columns':<10} | Notes")
print("-" * 80)
print(f"{'1. Raw Ingestion':<30} | {total_raw_rows:>13,} | {'Varies':<10} | {len(raw_dfs)} files loaded")
print(f"{'2. After Cleaning':<30} | {total_cleaned_rows:>13,} | {'Varies':<10} | Removed {total_raw_rows - total_cleaned_rows:,} bad rows")
print(f"{'3. After Transformation':<30} | {total_transformed_rows:>13,} | {'Varies':<10} | Column transformations applied")
print(f"{'4. After Merging':<30} | {merged_row_count:>13,} | {len(merged_df.columns):<10} | Unified schema")
print(f"{'5. After Dedup & QC':<30} | {final_row_count:>13,} | {len(final_df.columns):<10} | Removed {merged_row_count - final_row_count:,} rows")

print("\n" + "="*80)
print(f"Total rows removed from pipeline: {total_raw_rows - final_row_count:,}")
print(f"Data reduction: {((total_raw_rows - final_row_count) / total_raw_rows * 100):.2f}%")
print("="*80 + "\n")

# ── Save as Parquet ──

In [0]:

PARQUET_PATH = OUTPUT_P_PATH

final_df.write.mode("overwrite").parquet(PARQUET_PATH)

print(f"✓ Parquet saved to {PARQUET_PATH}")

#Viewing final DataFrame

In [0]:
final_df.display()

In [0]:
final_df.select("price_usd").distinct().display()
